In [6]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {
    'G': '\033[92m',   
    'R': '\033[91m',   
    'Y': '\033[93m',   
    'B': '\033[94m',   
    'C': '\033[96m',   
    'W': '\033[0m'     

STOP_LOSS_PCT = -0.01 
TRAILING_STOP_PCT = -0.015  
def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    print(f'Logged: {message}')
    
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'My passcode')
        smtp.send_message(msg)
last_buy_price = 0
last_signal = 'None'
highest_price = 0

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
                    highest_price = last_buy_price
        print(f' Memory recovered. Last signal: {last_signal}')
    except Exception as e:
        print(f' Memory recovery failed: {e}')

print(' Quantum Engine: Online')
while True:
    try:
        data = None
        while data is None:
            try:
                data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
                if data.empty:
                    data = None
                    raise ValueError('Data is empty')
            except Exception as e:
                log_system_event(f"Internet Hiccup: {e}. Retrying in 30s...")
                time.sleep(30)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        close_prices = data['Close']['RELIANCE.NS']
        data['Delta'] = close_prices.diff()
        data['Gain'] = data['Delta'].clip(lower=0)
        data['Loss'] = data['Delta'].clip(upper=0).abs()
        data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
        data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
        data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
        data['RSI'] = 100 - (100 / (1 + data['RS']))
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price 
            if highest_price == 0 or current_price > highest_price:
                highest_price = current_price
            drawdown_from_peak = (current_price - highest_price) / highest_price
            trigger_signal = None
            if current_pnl_pct <= STOP_LOSS_PCT:
                trigger_signal = 'STOP_LOSS'
            elif drawdown_from_peak <= TRAILING_STOP_PCT:
                trigger_signal = 'TRAILING_STOP' 
            if trigger_signal:
                pnl = current_price - last_buy_price
                print(f'\n {trigger_signal} HIT! Selling at {current_price:.2f}')
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                try:
                    send_trade_notification(trigger_signal, current_price, pnl)
                except Exception as e:
                    print(f'Notification Failed: {e}')
                last_signal = 'COOLDOWN'
                last_buy_price = 0
                highest_price = 0 
                time.sleep(60)
                continue 
        if sma_45_val > sma_190_val and current_rsi < 60:
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'
        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print('\n Cooldown finished. System reset.')
                last_signal = 'SELL'
            else:
                print(f'{time.ctime()} | Status: COOLDOWN')
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                last_buy_price = 0 
                highest_price = 0
            if current_signal == 'BUY':
                last_buy_price = current_price
                highest_price = current_price 
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
            send_trade_notification(current_signal, current_price, pnl)
            last_signal = current_signal
        else:
            pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
            rsi_display = f"{current_rsi:.2f}" if not pd.isna(current_rsi) else "N/A"
            trail_display = f" | Peak: {highest_price:.2f}" if last_signal == 'BUY' else ""
            print(f'{time.ctime()} | Price: {current_price:.2f} | RSI: {rsi_display} | Signal: {last_signal}{trail_display}')
    except Exception as e:
        error_msg = f'CRITICAL ERROR: {e}'
        print(f'\n {error_msg}')
        log_system_event(error_msg)
        try:
            send_trade_notification('System Alert', 0, 0)
        except:
            pass
        print('Restarting loop in 15 sec...')
        time.sleep(15)
        continue  
    time.sleep(60)

🤖 Memory recovered. Last signal: STOP_LOSS
🚀 Quantum Engine: Online
Mon May 11 16:14:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:15:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:16:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:17:16 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:18:16 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:19:17 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:20:17 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:21:18 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:22:18 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:23:18 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:24:19 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:25:19 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:26:20 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:27:20 20

KeyboardInterrupt: 

In [2]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {
    'G': '\033[92m',  
    'R': '\033[91m',  
    'Y': '\033[93m',  
    'B': '\033[94m',  
    'C': '\033[96m',  
    'W': '\033[0m'    
}
STOP_LOSS_PCT = -0.01 
TRAILING_STOP_PCT = -0.015  

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')   
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'My Passcode')
        smtp.send_message(msg)
last_buy_price = 0
last_signal = 'None'
highest_price = 0
total_pnl = 0.0
wins = 0
losses = 0
if os.path.exists('trade_log.csv'):
    try:
        df_history = pd.read_csv('trade_log.csv', names=['Time', 'Price', 'Signal', 'PnL'])
        df_history['PnL'] = pd.to_numeric(df_history['PnL'], errors='coerce').fillna(0)
        total_pnl = df_history['PnL'].sum()
        wins = len(df_history[df_history['PnL'] > 0])
        losses = len(df_history[df_history['PnL'] < 0])
        last_signal = str(df_history.iloc[-1]['Signal']).strip()
        if last_signal == 'BUY':
            last_buy_price = float(df_history.iloc[-1]['Price'])
            highest_price = last_buy_price
        print(f"{C['B']} Metrics Recovered: PnL: {total_pnl:.2f} | Wins: {wins} | Losses: {losses}{C['W']}")
    except Exception as e:
        print(f"{C['Y']} Metrics recovery note (Starting fresh): {e}{C['W']}")
print(f"{C['G']} Quantum Engine: Online (Command Center UI Active){C['W']}\n")
while True:
    try:
        data = None
        while data is None:
            try:
                data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
                if data.empty:
                    data = None
                    raise ValueError('Data is empty')
            except Exception as e:
                log_system_event(f"Internet Hiccup: {e}. Retrying...")
                print(f"\n{C['Y']} Connection interrupted. Retrying in 30s...{C['W']}")
                time.sleep(30)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        close_prices = data['Close']['RELIANCE.NS']
        data['Delta'] = close_prices.diff()
        data['Gain'] = data['Delta'].clip(lower=0)
        data['Loss'] = data['Delta'].clip(upper=0).abs()
        data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
        data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
        data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
        data['RSI'] = 100 - (100 / (1 + data['RS']))
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price 
            if highest_price == 0 or current_price > highest_price:
                highest_price = current_price
            drawdown_from_peak = (current_price - highest_price) / highest_price
            trigger_signal = None
            if current_pnl_pct <= STOP_LOSS_PCT:
                trigger_signal = 'STOP_LOSS'
            elif drawdown_from_peak <= TRAILING_STOP_PCT:
                trigger_signal = 'TRAILING_STOP' 
            if trigger_signal:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                print(f"\n{C['R']} {trigger_signal} HIT! Selling at {current_price:.2f} | Trade PnL: {pnl:.2f}{C['W']}")
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                try: send_trade_notification(trigger_signal, current_price, pnl)
                except: pass
                last_signal = 'COOLDOWN'
                last_buy_price = 0
                highest_price = 0 
                time.sleep(60)
                continue 
        if sma_45_val > sma_190_val and current_rsi < 60:
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'
        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print(f"\n{C['C']} Cooldown finished. System reset.{C['W']}")
                last_signal = 'SELL'
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                print(f"\n{C['Y']} Trade Closed at {current_price:.2f} | PnL: {pnl:.2f}{C['W']}")
                last_buy_price = 0 
                highest_price = 0
            if current_signal == 'BUY':
                last_buy_price = current_price
                highest_price = current_price 
                print(f"\n{C['G']} ENTRY ALERT: Buying at {current_price:.2f}{C['W']}")
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
            try: send_trade_notification(current_signal, current_price, pnl)
            except: pass
            last_signal = current_signal
        pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
        rsi_color = C['Y'] if current_rsi > 60 or current_rsi < 30 else C['C']
        pnl_color = C['G'] if pnl_display >= 0 else C['R']
        total_pnl_color = C['G'] if total_pnl >= 0 else C['R']
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | "
              f"RSI: {rsi_color}{current_rsi:.1f}{C['W']} | "
              f"Open PnL: {pnl_color}{pnl_display:+.2f}{C['W']} | "
              f"Total PnL: {total_pnl_color}{total_pnl:+.2f}{C['W']} | "
              f"W/L: {wins}/{losses} "
              f"({last_signal})    ", end="")
    except Exception as e:
        error_msg = f'CRITICAL ERROR: {e}'
        print(f"\n{C['R']} {error_msg}{C['W']}")
        log_system_event(error_msg)
        time.sleep(15)
        continue  

    time.sleep(60)

 Metrics Recovered: PnL: -63.30 | Wins: 0 | Losses: 3
 Quantum Engine: Online (Command Center UI Active)

[18:26:17] Prc: 1366.90 | RSI: 73.8 | Open PnL: +0.00 | Total PnL: -63.30 | W/L: 0/3 (SELL)    

KeyboardInterrupt: 